# Week 7 – Multi-task + RAG (one file)

**Two ideas:**
1. **Multi-task**: Predict category + price in one completion (`Category: X. Price is $Y.00`).
2. **RAG**: Compare **pricer only** vs **pricer + RAG** (similar products in prompt); report MAE.

Run from repo root or copy to Google Colab. All code is in this notebook.

In [14]:
# Run once if you get ModuleNotFoundError: No module named 'peft'
%pip install -q peft transformers accelerate bitsandbytes

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
293.27s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Setup and path

In [15]:
import sys
from pathlib import Path
def find_week7_root():
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / "week7" / "pricer").exists():
            return p / "week7"
        if (p / ".git").exists() and (p / "week7").exists():
            return p / "week7"
        p = p.parent
    return Path.cwd() / "week7" if (Path.cwd() / "week7" / "pricer").exists() else Path.cwd()
week7_dir = find_week7_root()
sys.path.insert(0, str(week7_dir))

import os
import re
from typing import List, Optional
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.items import Item
from pricer.evaluator import evaluate
from tqdm.notebook import tqdm
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

## Helpers (multi-task + RAG, inline)

In [16]:
import numpy as np

MULTITASK_QUESTION = (
    "What category is this product and what does it cost to the nearest dollar? "
    "Reply with: Category: <category>. Price is $<number>."
)
MULTITASK_PREFIX = "Category: "

def build_multitask_prompt(summary: str) -> str:
    return f"{MULTITASK_QUESTION}\n\n{summary}\n\n{MULTITASK_PREFIX}"

def build_multitask_completion(category: str, price: float, do_round: bool = True) -> str:
    p = round(price) if do_round else price
    return f"{category}. Price is ${p}.00"

def parse_multitask_completion(completion: str) -> tuple[Optional[str], float]:
    completion = (completion or "").strip()
    price_match = re.search(r"Price is\s*\$?\s*([-+]?\d*\.?\d+)", completion, re.I)
    price = float(price_match.group(1)) if price_match else 0.0
    category = None
    if "Price is" in completion:
        left = completion.split("Price is")[0].strip()
        if left.lower().startswith("category:"): left = left[9:].strip()
        if left.endswith("."): left = left[:-1].strip()
        if left: category = left
    return (category, price)

def item_to_multitask_datapoint(item, tokenizer, max_tokens: int, do_round: bool = True) -> dict:
    summary = getattr(item, "summary", None) or ""
    tokens = tokenizer.encode(summary, add_special_tokens=False)
    if len(tokens) > max_tokens:
        summary = tokenizer.decode(tokens[:max_tokens]).rstrip()
    return {"prompt": build_multitask_prompt(summary), "completion": build_multitask_completion(getattr(item, "category", ""), getattr(item, "price", 0), do_round)}

In [4]:
_emb_model = None

def _get_emb():
    global _emb_model
    if _emb_model is None:
        from sentence_transformers import SentenceTransformer
        _emb_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    return _emb_model

def embed_texts(texts: List[str]) -> np.ndarray:
    return np.array(_get_emb().encode(texts, show_progress_bar=len(texts) > 50), dtype="float32")

class RAGIndex:
    def __init__(self, items: list):
        self.items = items
        self.texts = [(i.summary or "") if hasattr(i, "summary") else str(i) for i in items]
        self._emb = embed_texts(self.texts)

    def format_similar_products(self, query: str, k: int = 5) -> str:
        q = np.array(_get_emb().encode([query], show_progress_bar=False), dtype="float32")
        dists = np.linalg.norm(self._emb - q, axis=1)
        top_idx = np.argsort(dists)[:k]
        parts = [f'"{getattr(self.items[i], "title", "Product") or "Product"[:50]}" ${getattr(self.items[i], "price", 0):.0f}' for i in top_idx]
        return "Similar products: " + "; ".join(parts) if parts else "Similar products: (none)"

def build_rag_index(train_items: list) -> RAGIndex:
    return RAGIndex(train_items)

## Load data

In [5]:
load_dotenv(override=True)
if os.environ.get("HF_TOKEN"):
    login(os.environ["HF_TOKEN"], add_to_git_credential=True)

LITE_MODE = True
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"
train, val, test = Item.from_hub(dataset)
print(f"Train {len(train)}, Val {len(val)}, Test {len(test)}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Train 20000, Val 1000, Test 1000


## Multi-task dataset (prompt/completion)

In [6]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
CUTOFF = 110
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

train_dp = [item_to_multitask_datapoint(i, tokenizer, CUTOFF, do_round=True) for i in tqdm(train)]
val_dp = [item_to_multitask_datapoint(i, tokenizer, CUTOFF, do_round=True) for i in tqdm(val)]
test_dp = [item_to_multitask_datapoint(i, tokenizer, CUTOFF, do_round=False) for i in tqdm(test)]
print("Example completion:", train_dp[0]["completion"])

  0%|          | 0/20000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

Example completion: Tools_and_Home_Improvement. Price is $64.00


In [7]:
MULTITASK_DATASET = None  # e.g. "abdussamadbello/items_prompts_multitask_lite"
if MULTITASK_DATASET:
    DatasetDict({"train": Dataset.from_list(train_dp), "validation": Dataset.from_list(val_dp), "test": Dataset.from_list(test_dp)}).push_to_hub(MULTITASK_DATASET)
    print(f"Pushed to {MULTITASK_DATASET}")

## Model and predictors (pricer only vs pricer + RAG)

In [ ]:
ADAPTER_PATH = "ed-donner/pricer-2024-09-13_13.04.39"  # ed-donner's fine-tuned pricer

def make_generate_fn():
    if not ADAPTER_PATH:
        return None
    try:
        import accelerate
    except ModuleNotFoundError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "accelerate"])
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftConfig, PeftModel
    import torch

    adapter_cfg = PeftConfig.from_pretrained(ADAPTER_PATH)
    inference_base_model = adapter_cfg.base_model_name_or_path or BASE_MODEL
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

    model = AutoModelForCausalLM.from_pretrained(
        inference_base_model,
        quantization_config=bnb,
        device_map="auto",
    )
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    tok = AutoTokenizer.from_pretrained(inference_base_model)

    def gen(prompt: str, max_new_tokens=80):
        inp = tok(prompt, return_tensors="pt").to(model.device)
        out = model.generate(
            **inp,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
        return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

    return gen

generate_fn = make_generate_fn()
RAG_K = 5
rag_index = build_rag_index(train)


ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

In [10]:
def prompt_with_rag(item, with_rag: bool):
    summary = item.summary or ""
    base = build_multitask_prompt(summary)
    if not with_rag:
        return base
    return rag_index.format_similar_products(summary, k=RAG_K) + "\n\n" + base

def pricer_only(item):
    prompt = prompt_with_rag(item, with_rag=False)
    if generate_fn is None: return item.price
    _, price = parse_multitask_completion(generate_fn(prompt))
    return price

def pricer_with_rag(item):
    prompt = prompt_with_rag(item, with_rag=True)
    if generate_fn is None: return item.price
    _, price = parse_multitask_completion(generate_fn(prompt))
    return price

## Compare pricer only vs pricer + RAG

In [12]:
EVAL_SIZE = 50
if generate_fn:
    print("=== Pricer only ===")
    evaluate(pricer_only, test, size=EVAL_SIZE)
    print("=== Pricer + RAG ===")
    evaluate(pricer_with_rag, test, size=EVAL_SIZE)
else:
    print("Set ADAPTER_PATH to run comparison.")

Set ADAPTER_PATH to run comparison.
